# 03 — Hugging Face Embeddings and Semantic Retrieval

This notebook implements and evaluates the **retrieval** part of RAG. It does not generate an answer yet. Keeping retrieval separate lets us measure whether the correct evidence reaches the future LLM.

## Architecture

**Offline indexing:** cleaned PDF pages → token-aware chunks → Hugging Face SentenceTransformer → normalized 384-dimensional vectors.

**Online retrieval:** user question → the same SentenceTransformer → normalized query vector → cosine similarity against every chunk → Top-k evidence.

> Key implementation: `SentenceTransformer.encode(..., normalize_embeddings=True)`. Because document and query vectors have length 1, cosine similarity becomes the dot product `embeddings @ query_vector`.

In [1]:
from pathlib import Path
import json
import sys
import os
import logging
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
logging.getLogger('tensorflow').setLevel(logging.ERROR)
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.embed_retrieve import (
    read_jsonl, load_embedding_model, validate_embeddings,
    embed_query, retrieve_top_k, evaluate_retrieval
)

MODEL_PATH = PROJECT_ROOT / 'models' / 'all-MiniLM-L6-v2'
model = load_embedding_model(MODEL_PATH, device='cpu')
print('Model:', MODEL_PATH.name)
print('Maximum input length:', model.max_seq_length, 'tokens')
print('Embedding dimension:', model.get_sentence_embedding_dimension())

Model: all-MiniLM-L6-v2
Maximum input length: 256 tokens
Embedding dimension: 384


## Checkpoint 1 — Load vectors and preserve row alignment

`embeddings[i]` must always describe `chunks[i]`. If rows and metadata become misaligned, retrieval can return the right vector but cite the wrong text. The manifest records the source chunk file hash and boundary IDs; the validation below checks shape, finiteness, dimension, normalization, and row count.

In [2]:
artifacts = {}
for size in (128, 220):
    chunks = read_jsonl(PROJECT_ROOT / f'data/processed/chunks_{size}.jsonl')
    embeddings = np.load(PROJECT_ROOT / f'data/embeddings/embeddings_{size}.npy')
    report = validate_embeddings(embeddings, chunks)
    artifacts[size] = {'chunks': chunks, 'embeddings': embeddings}
    print(f'{size}-token design: shape={embeddings.shape}, normalized={report["normalized"]}, row_alignment={report["row_alignment_valid"]}')

128-token design: shape=(972, 384), normalized=True, row_alignment=True
220-token design: shape=(544, 384), normalized=True, row_alignment=True


## Checkpoint 2 — The essential embedding implementation

The complete corpus is embedded in batches to control memory. `convert_to_numpy=True` produces a matrix for fast numerical search, while `normalize_embeddings=True` makes similarity calculation simple and consistent.

```python
texts = [chunk["text"] for chunk in chunks]
embeddings = model.encode(
    texts, batch_size=32, convert_to_numpy=True,
    normalize_embeddings=True
)
```

The same model must encode both documents and questions. Vectors from unrelated models do not share a meaningful coordinate space.

## Checkpoint 3 — Query encoding and exact semantic search

For this small corpus, exact NumPy search is transparent and fast. A vector database such as FAISS becomes useful when scanning all vectors is too expensive. The score below is cosine similarity, **not a probability or confidence percentage**.

In [3]:
question = 'What measurements and information support predictive maintenance decisions?'
query_vector = embed_query(model, question)
scores = artifacts[128]['embeddings'] @ query_vector
print('Query vector shape:', query_vector.shape)
print('Query vector norm:', round(float(np.linalg.norm(query_vector)), 6))
print('Similarity array shape:', scores.shape)

results = retrieve_top_k(question, model, artifacts[128]['embeddings'], artifacts[128]['chunks'], top_k=3)
for result in results:
    print('\n', '=' * 100)
    print(f"Rank {result['rank']} | cosine={result['score']:.3f} | {result['document']} | PDF page {result['pdf_page']}")
    print(result['text'][:700])

Query vector shape: (384,)
Query vector norm: 1.0
Similarity array shape: (972,)

Rank 1 | cosine=0.722 | NIST.AMS.100-18 | PDF page 20
Meanwhile only a third of those that focus on quality have a high level (see Table 2-3).
Approximately 52 % of companies that focus on quality indicated that they have a high level of predictive maintenance.
Moreover, Table 2-3 shows that cost competitive companies along with those focusing on flexibility tend to favor reactive maintenance while those pursuing quality as a competitive priority favor preventive and predictive maintenance.
Jin et al (2017a and 2017b) found in a survey that companies are starting to consider predictive maintenance techniques with a majority of their respondents having active projects in manufacturing diagnostics and prognostics.

Rank 2 | cosine=0.719 | NIST.AMS.100-18 | PDF page 7
The potential effect on maintenance costs from adopting predictive maintenance techniques is not well documented at the national level.
The es

## Checkpoint 4 — Evaluate retrieval before adding an LLM

Ten questions were manually labelled with acceptable document/page evidence.

- **Hit@k:** fraction of questions with acceptable evidence in the first k results.
- **MRR:** mean reciprocal rank; rewards placing the first acceptable result nearer rank 1.

This evaluates retrieval only. Answer quality must later be evaluated separately because a generator can still misinterpret good context.

In [4]:
questions = json.loads((PROJECT_ROOT / 'evaluation/retrieval_questions.json').read_text(encoding='utf-8'))
evaluation_rows = {}
summaries = []

for size in (128, 220):
    rows, summary = evaluate_retrieval(
        questions, model, artifacts[size]['embeddings'], artifacts[size]['chunks'], search_k=10
    )
    evaluation_rows[size] = rows
    summaries.append({'chunk_tokens': size, **summary})

summary_df = pd.DataFrame(summaries)
summary_df.style.format({
    'hit_at_1': '{:.0%}', 'hit_at_3': '{:.0%}',
    'hit_at_5': '{:.0%}', 'mean_reciprocal_rank': '{:.3f}'
})

,chunk_tokens,questions,hit_at_1,hit_at_3,hit_at_5,mean_reciprocal_rank
0,128,10,50%,90%,90%,0.633
1,220,10,50%,70%,80%,0.619


In [5]:
detail_128 = pd.DataFrame(evaluation_rows[128])[
    ['question_id', 'first_relevant_rank', 'hit_at_3', 'top_result_document', 'top_result_page', 'top_result_score']
]
detail_128.style.format({'top_result_score': '{:.3f}'})

,question_id,first_relevant_rank,hit_at_3,top_result_document,top_result_page,top_result_score
0,Q01,3.000000,True,NIST.AMS.100-34,47,0.778
1,Q02,1.000000,True,NIST.AMS.100-34,15,0.677
2,Q03,3.000000,True,NIST.AMS.100-18,20,0.777
3,Q04,nan,False,NIST.AMS.100-34,5,0.657
4,Q05,1.000000,True,NIST.AMS.100-2,33,0.756
5,Q06,1.000000,True,NIST.AMS.100-18,17,0.806
6,Q07,1.000000,True,NIST.AMS.100-34,22,0.774
7,Q08,3.000000,True,NIST.AMS.100-18,17,0.738
8,Q09,1.000000,True,NIST.AMS.100-2,56,0.678
9,Q10,3.000000,True,NIST.AMS.100-34,7,0.755


## Checkpoint 5 — Inspect a failure, then make the design decision

The 128-token configuration has one question whose labelled page does not appear in the first 10 results. We inspect its retrieved text to decide whether the issue is poor retrieval, incomplete labels, terminology mismatch, or a genuinely ambiguous question. This is more useful than tuning only for a headline metric.

In [6]:
failed = next(row for row in evaluation_rows[128] if row['first_relevant_rank'] is None)
print('Failure:', failed['question_id'], '-', failed['question'])
failure_results = retrieve_top_k(
    failed['question'], model, artifacts[128]['embeddings'], artifacts[128]['chunks'], top_k=3
)
for result in failure_results:
    print('\n', '-' * 100)
    print(f"Rank {result['rank']} | cosine={result['score']:.3f} | {result['document']} p{result['pdf_page']}")
    print(result['text'][:600])

Failure: Q04 - What is a disadvantage of performing excessive preventive maintenance?

 ----------------------------------------------------------------------------------------------------
Rank 1 | cosine=0.657 | NIST.AMS.100-34 p5
29
Perceived Benefits of Advancing Maintenance Strategies .........................................
32
Observed Benefits of Preventive and Predictive Maintenance .................................
35
7.1.

 ----------------------------------------------------------------------------------------------------
Rank 2 | cosine=0.653 | NIST.AMS.100-34 p13
maintenance while Section 7 highlights observed differences between firms that have adopted more preventive and predictive maintenance.
Finally, Section 8 summarizes the findings.

 ----------------------------------------------------------------------------------------------------
Rank 3 | cosine=0.639 | NIST.AMS.100-34 p46
The implication being that reactive maintenance reduces quality and increases uncertainty 

## Engineering conclusion

Use the **128-token corpus** for the first RAG version: it achieved stronger Hit@3 and MRR on this small labelled set. The likely explanation is that smaller chunks contain less unrelated text, so their vector represents a more focused topic. The result is preliminary because ten questions are not enough for a production claim.

### What you should be able to explain in an interview

1. Why a sentence embedding represents meaning rather than exact keyword matching.
2. Why document chunks and queries must use the same Hugging Face model.
3. Why normalized vectors let cosine similarity use a dot product.
4. Why metadata-to-vector row alignment is a correctness requirement.
5. Why exact NumPy search is appropriate for 972 vectors, and when FAISS/vector databases become necessary.
6. Why retrieval is evaluated before answer generation.
7. Why the measured comparison supports selecting 128-token chunks without claiming universal superiority.

**Next RAG stage:** format the Top-k chunks as grounded context, design a prompt that requires citations and permits an ‘insufficient evidence’ response, then connect an LLM and evaluate faithfulness.